# Batch Process Graphs with Hierarchical DP Merge Ordering

This notebook processes all `.gpickle` graph files in the directory structure, applies the hierarchical DP merge ordering (Peeling + Reverse Cuthill-McKee + Reconciliation), and saves the statistics (number of emitters and emitter CNOTs).

In [5]:
import glob
import os
import re
import csv
import time
import pickle
import networkx as nx
import numpy as np
from networkx.utils import reverse_cuthill_mckee_ordering
from optimization_script import run_optimization
from lib.circuitSolver import *
from lib.generate_graph import *
from lib.tableau import *
from clustering import GreedyClustering, KernighanLinClustering
import matplotlib.pyplot as plt
from clustering import Cluster

In [ ]:
# --- Define Egde reduction function ---
def edge_reduction_func(input_graph):
    our_graph_original = input_graph.copy()
    # Create a mapping dictionary: (0, 0) -> 0, (0, 1) -> 1, ..., (2, 2) -> 8
    mapping = {node: i for i, node in enumerate(our_graph_original.nodes())}

    # Create a new graph with integer labels
    our_graph = nx.relabel_nodes(our_graph_original, mapping)
    graph , circ, statistics = run_optimization(our_graph, edge_reduction=True, minLA=False, opt_CNOT= False)
    return graph

# --- Define the DP Ordering Functions ---

def count_active_cross_edges(graph, set_A_remaining, set_B_remaining):
    """
    Helper to count edges between A_remaining and B_remaining.
    (Optimized for clarity, can be cached for performance)
    """
    count = 0
    # This is the naive O(N^2) check. 
    # For strict Rank-1 cuts, this is simply len(connectors_A_in_rem) * len(connectors_B_in_rem)
    for u in set_A_remaining:
        for v in set_B_remaining:
            if graph.has_edge(u, v):
                count += 1
    return count

def dp_merge_orders(graph, order_A, order_B):
    """
    Merges two ordered lists of nodes (order_A, order_B) to minimize 
    Linear Arrangement cost of edges BETWEEN A and B.
    
    This is equivalent to finding the shortest path in a grid.
    """
    n = len(order_A)
    m = len(order_B)
    
    # DP State: dp[i][j] = min cumulative edge length using first i of A and j of B
    dp = np.full((n + 1, m + 1), np.inf)
    parent = {} # To reconstruct the path: (i, j) -> (prev_i, prev_j, 'A' or 'B')
    
    dp[0][0] = 0
    
    # Pre-calculate active edges for performance (Optional but recommended)
    # Ideally, we know the "connector" nodes. 
    # For now, we compute cost on the fly:
    # Cost added at step k = number of "open" edges crossing the cut.
    # Open edges = edges between (Placed_A, Remaining_B) and (Placed_B, Remaining_A)
    
    # We iterate through the grid
    for i in range(n + 1):
        for j in range(m + 1):
            if i == 0 and j == 0:
                continue
            
            # Current sets of REMAINING nodes (not yet placed)
            # If we are at state (i, j), it means order_A[:i] and order_B[:j] are placed.
            # The "cost" of the current state is the number of active edges 
            # stretching across the future positions.
            
            rem_A = order_A[i:]
            rem_B = order_B[j:]
            placed_A = order_A[:i]
            placed_B = order_B[:j]
            
            # Calculate cross-cut active edges (this is the weight of being at state i,j)
            # Rank 1 Optimization: We only care about edges between A and B.
            # Edges internal to A or B are already fixed by recursive calls.
            
            # Note: This cost calculation is the "Cut Width" at this position.
            # Sum of Cut Widths = Total Edge Length.
            
            # Bridges stretching from Placed A -> Remaining B
            c1 = 0
            for u in placed_A:
                for v in rem_B:
                    if graph.has_edge(u, v): c1 += 1
            
            # Bridges stretching from Placed B -> Remaining A
            c2 = 0
            for u in placed_B:
                for v in rem_A:
                    if graph.has_edge(u, v): c2 += 1
            
            current_cut_cost = c1 + c2
            
            # Transitions
            # Coming from picking A[i-1] (Moving down)
            if i > 0:
                prev_cost = dp[i-1][j]
                if prev_cost + current_cut_cost < dp[i][j]:
                    dp[i][j] = prev_cost + current_cut_cost
                    parent[(i,j)] = (i-1, j, 'A')
            
            # Coming from picking B[j-1] (Moving right)
            if j > 0:
                prev_cost = dp[i][j-1]
                if prev_cost + current_cut_cost < dp[i][j]:
                    dp[i][j] = prev_cost + current_cut_cost
                    parent[(i,j)] = (i, j-1, 'B')

    # Backtrack to find the merge
    merged_order = []
    curr = (n, m)
    while curr != (0, 0):
        prev_i, prev_j, choice = parent[curr]
        if choice == 'A':
            # We picked A[i-1]
            merged_order.append(order_A[curr[0]-1])
        else:
            merged_order.append(order_B[curr[1]-1])
        curr = (prev_i, prev_j)
    
    return merged_order[::-1] # Reverse because we backtracked


def hierarchical_ordering(graph, cluster):
    """
    Recursive function replacing 'peel_to_rank1'.
    1. If cluster is leaf/rank1: Solve RCM.
    2. Else: Recurse children, then MERGE their orders.
    """
    # Base Case: Stop if cutrank is 1 OR no children
    if cluster.cutrank <= 1 or cluster.inherited is None:
        # Apply RCM to this leaf cluster
        if len(cluster.nodes) == 0: return []
        subgraph = graph.subgraph(cluster.nodes)
        try:
            # RCM
            return list(reverse_cuthill_mckee_ordering(subgraph))
        except:
            return list(cluster.nodes)
            
    # Recursive Step
    # Note: greedy clustering usually produces 2 children (binary tree)
    # But inherited is a list, so we fold them.
    
    child_orders = []
    for child in cluster.inherited:
        child_orders.append(hierarchical_ordering(graph, child))
    
    # Merge children sequentially (if > 2, merge 1 and 2, then result with 3...)
    final_order = child_orders[0]
    for next_order in child_orders[1:]:
        final_order = dp_merge_orders(graph, final_order, next_order)
        
    return final_order

#data analysis functions
def find_saved_graph(node_number, p, idx=None, search_root='.'):
    """Return path to a saved graph matching node_number and p (or None)."""
    p_safe = str(p)
    patterns = [
        f"**/*graph_n{node_number}_p{p_safe}_idx*.gpickle",
        f"**/*_graph_n{node_number}_p{p_safe}_idx*.gpickle",
        f"**/*p{p_safe}*/**/*graph_n{node_number}_p{p_safe}_idx*.gpickle",
        f"**/*p{p_safe}*/**/*_graph_n{node_number}_p{p_safe}_idx*.gpickle"
    ]
    matches = []
    for pat in patterns:
        matches.extend(glob.glob(os.path.join(search_root, pat), recursive=True))
    matches = sorted(set(matches))
    if not matches:
        return None
    if idx is not None:
        candidates = [m for m in matches if f"_idx{idx}.gpickle" in m]
        if candidates:
            return candidates[0]
    return matches[0]

def load_graph_from_file(path):
    """Load NetworkX graph from a .gpickle or pickled file."""
    try:
        return nx.read_gpickle(path)
    except Exception:
        with open(path, 'rb') as f:
            return pickle.load(f)


In [4]:
# Batch-process saved graphs: compute number of emitters and emitter CNOTs for each saved graph (incremental save)
import glob
import os
import re
import csv
import time
from pathlib import Path

# Where to search for saved graphs (relative to notebook location)
search_root = '.'
# patterns that match both old and new naming conventions
patterns = [
    os.path.join(search_root, '**', 'graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '_graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*_graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*_idx*.gpickle')
]

# gather candidate files
files = []
for pat in patterns:
    files.extend(sorted(glob.glob(pat, recursive=True)))
# filter to .gpickle only and remove duplicates
files = sorted({f for f in files if f.lower().endswith('.gpickle')})
if not files:
    print('No .gpickle files found under', search_root)
else:
    print(f'Found {len(files)} candidate .gpickle files; processing...')

# output CSV path (incrementally appended)
out_csv = os.path.join(search_root, 'graph_stats_dp_ordering.csv')
fieldnames = ['path','node_number','p','idx','num_emitters','num_emitter_CNOTs']
# ensure header exists (create file if missing)
if not os.path.exists(out_csv):
    with open(out_csv, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

start = time.time()
for i, path in enumerate(files):
    print(f'[{i+1}/{len(files)}] Processing: {path}')
    try:
        # attempt to extract metadata from filename
        # look for graph_n{n}_p{p}_idx{k} pattern
        m = re.search(r'graph_n(\d+)_p([0-9\.]+)_idx(\d+)', path)
        if m:
            node_n = int(m.group(1))
            p_val = float(m.group(2))
            idx_val = int(m.group(3))
        else:
            # try to fall back to other filename formats
            m2 = re.search(r'_idx(\d+)\.gpickle', path)
            idx_val = int(m2.group(1)) if m2 else None
            m3 = re.search(r'p([0-9\.]+)', path)
            p_val = float(m3.group(1)) if m3 else None
            m4 = re.search(r'graph_n(\d+)', path)
            node_n = int(m4.group(1)) if m4 else None
        # load graph (use helper if available)
        try:
            G = load_graph_from_file(path)
        except NameError:
            G = nx.read_gpickle(path)
        # run optimization (assumes run_optimization imported at top of notebook)
        try:
            G = edge_reduction_func(G)
            greedy = GreedyClustering()
            root_cluster = greedy.do(G)

            # This single call does the peeling, leaf ordering, AND DP reconciliation
            final_ordering_dp = hierarchical_ordering(G, root_cluster)

            print(f"Global Ordering (DP): {final_ordering_dp}")

            # Visualize and Test
            ordered_graph_dp = nx.relabel_nodes(G, {old: new for new, old in enumerate(final_ordering_dp)})
            opt_graph, circuit, stats = run_optimization(ordered_graph_dp, num_LA= 20)
        except Exception as e:
            print('run_optimization failed for', path, '->', e)
            stats = None
        if stats is None:
            num_emitters = None
            num_emitter_CNOTs = None
        else:
            # stats expected to be an iterable with at least two entries
            num_emitters = stats[0] if len(stats) > 0 else None
            num_emitter_CNOTs = stats[1] if len(stats) > 1 else None
        print(f'  -> emitters={num_emitters}, emitter_CNOTs={num_emitter_CNOTs}')
    except Exception as e:
        print('  ERROR processing', path, ':', e)
        node_n = None
        p_val = None
        idx_val = None
        num_emitters = None
        num_emitter_CNOTs = None
    # write row immediately so results are visible during long runs
    row = {
        'path': path,
        'node_number': node_n,
        'p': p_val,
        'idx': idx_val,
        'num_emitters': num_emitters,
        'num_emitter_CNOTs': num_emitter_CNOTs
    }
    # append to CSV (file opened and closed per row to ensure data flushed to disk)
    try:
        with open(out_csv, 'a', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writerow(row)
    except Exception as e:
        print('  ERROR writing row to CSV:', e)

elapsed = time.time() - start
print(f'Done. Processed {len(files)} files in {elapsed:.1f}s. Results saved (incrementally) to {out_csv}')

Found 3120 candidate .gpickle files; processing...
[1/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx1.gpickle
Global Ordering (DP): [0, 5, 2, 1, 4, 3]


/Users/konstantinosrafailrevis/Desktop/PhD/0paul_state_gen/photonic-graphstate-generator-main/lib/generate_graph.py:133: RuntimeWarning: overflow encountered in divide
  if next_cost < current_cost or random.random() < np.exp((current_cost - next_cost)/temperature):


  -> emitters=1, emitter_CNOTs=0
[2/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx10.gpickle
Global Ordering (DP): [5, 4, 2, 3, 0, 1]
  -> emitters=1, emitter_CNOTs=0
[3/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx2.gpickle
Global Ordering (DP): [2, 3, 4, 1, 0, 5]
  -> emitters=1, emitter_CNOTs=0
[4/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx3.gpickle
Global Ordering (DP): [3, 1, 0, 5, 2, 4]
  -> emitters=1, emitter_CNOTs=0
[5/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx4.gpickle
Global Ordering (DP): [5, 1, 3, 4, 0, 2]
  -> emitters=1, emitter_CNOTs=0
[6/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx5.gpickle
Global Ordering (DP): [4, 3, 1, 0, 5, 2]
  -> emitters=1, emitter_CNOTs=0
[7/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx6.gpickle
Global Ordering (DP): [5, 3, 1, 0, 4, 2]
  -> emitters=1, emitter_CNOTs=0
[8/3120] Processing: ./graph_db_node_6/p_0_1/graph_n6_p0_1_idx7.gpickle
Global Ordering (D